# EDA — Survival interval diagnostics

Investigates why `ci_119` and `roc_auc_119` are degenerate in `train_lightning_model`
while `train_pycox_baseline` is unaffected.

Root hypothesis: evaluation time 119 is the last discrete bin. After discretization,
all patients have `disc_duration <= 118`, so `y_binary = (survival_times > 119)` is
all-zeros → AUC undefined → 0. CI degrades because `surv[:, 119]` collapses to 0.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from pathlib import Path

# ── adjust these paths ────────────────────────────────────────────────────────
PARQUET_DIR  = Path("data/dummy_data/longitudinal_dummy_smart_survival")  # output of preprocess_smartehr_survival.py
JSONL_DIR    = Path("data/dummy_data/longitudinal_smartehr_0_36500")       # output of smartehr_pipeline.py
NUM_TIME_INTERVALS = 120
HORIZON_DAYS       = 3650   # must match the --horizon-days used when running preprocess_smartehr_survival.py
EVALUATION_TIMES   = [12, 24, 36, 48, 60, 72, 84, 96, 108, 119]          # from survival_analysis_10yr.yaml
# ─────────────────────────────────────────────────────────────────────────────

## 1. Raw continuous durations from parquet

In [ ]:
splits = {}
for name in ["train", "validation", "test"]:
    df = pq.read_table(PARQUET_DIR / f"{name}.parquet").to_pandas()
    splits[name] = df
    print(f"{name:12s}: {len(df):,} rows | events={int(df['event'].sum())} ({100*df['event'].mean():.1f}%) "
          f"| duration min={df['duration'].min():.0f} max={df['duration'].max():.0f}")

In [ ]:
train = splits["train"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for event_val, label, color in [(1, "event", "tomato"), (0, "censored", "steelblue")]:
    mask = train["event"] == event_val
    axes[0].hist(train.loc[mask, "duration"], bins=60, alpha=0.6, label=f"{label} (n={mask.sum()})", color=color)
axes[0].axvline(HORIZON_DAYS, color="black", linestyle="--", label=f"horizon={HORIZON_DAYS}")
axes[0].set_title("Train: raw duration distribution")
axes[0].set_xlabel("days")
axes[0].legend()

# Zoom into the tail
tail = train[train["duration"] > train["duration"].quantile(0.90)]
for event_val, label, color in [(1, "event", "tomato"), (0, "censored", "steelblue")]:
    mask = tail["event"] == event_val
    axes[1].hist(tail.loc[mask, "duration"], bins=40, alpha=0.6, label=label, color=color)
axes[1].axvline(HORIZON_DAYS, color="black", linestyle="--", label=f"horizon={HORIZON_DAYS}")
axes[1].set_title("Train: duration tail (>p90)")
axes[1].set_xlabel("days")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nPatients at exactly horizon ({HORIZON_DAYS}d):")
for name, df in splits.items():
    at_horizon = (df["duration"] == HORIZON_DAYS).sum()
    print(f"  {name:12s}: {at_horizon:,} ({100*at_horizon/len(df):.1f}%)")

## 2. Simulate discretization — reproduce what utils.py does

In [ ]:
max_duration = float(train["duration"].max())
cuts = np.linspace(0, max_duration, NUM_TIME_INTERVALS + 1)[1:-1]  # 119 interior cuts

print(f"max_duration (from train): {max_duration:.1f} days")
print(f"Number of cut points:      {len(cuts)}")
print(f"cuts[0]  = {cuts[0]:.2f} days")
print(f"cuts[-1] = {cuts[-1]:.2f} days")
print(f"cuts[-1] == horizon? {cuts[-1] == HORIZON_DAYS}")
print()

def discretize(durations, cuts):
    clipped = np.clip(durations, 0, cuts[-1])
    return np.searchsorted(cuts, clipped).astype(np.int64)

for name, df in splits.items():
    disc = discretize(df["duration"].values, cuts)
    splits[name] = df.copy()
    splits[name]["disc_duration"] = disc
    print(f"{name:12s}: disc_duration min={disc.min()} max={disc.max()}")

## 3. Per-interval patient count and event rate

In [ ]:
val = splits["validation"]
interval_stats = []
for t in range(NUM_TIME_INTERVALS):
    mask = val["disc_duration"] == t
    n = mask.sum()
    n_events = val.loc[mask, "event"].sum() if n > 0 else 0
    interval_stats.append({"interval": t, "n_patients": n, "n_events": n_events,
                            "event_rate": n_events / n if n > 0 else np.nan})
stats_df = pd.DataFrame(interval_stats)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].bar(stats_df["interval"], stats_df["n_patients"], color="steelblue", alpha=0.7)
for t in EVALUATION_TIMES:
    axes[0].axvline(t, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Validation: patients per interval (red = evaluation times)")
axes[0].set_ylabel("count")

axes[1].bar(stats_df["interval"], stats_df["event_rate"], color="tomato", alpha=0.7)
for t in EVALUATION_TIMES:
    axes[1].axvline(t, color="black", linestyle="--", alpha=0.5)
axes[1].set_title("Validation: event rate per interval (black = evaluation times)")
axes[1].set_ylabel("event rate")
axes[1].set_xlabel("interval index")

plt.tight_layout()
plt.show()

print("\nLast 5 intervals:")
print(stats_df.tail(5).to_string(index=False))
print("\nAt evaluation times:")
print(stats_df[stats_df["interval"].isin(EVALUATION_TIMES)].to_string(index=False))

## 4. Why roc_auc_119 = 0 — reproduce the AUC mask logic

`time_dependent_roc_auc_score` at time `t` does:
1. `mask = ~((event==0) & (survival_times <= t))` — exclude censored patients whose outcome at t is unknown
2. `y_binary = (survival_times[mask] > t)` — 1 = alive at t (control), 0 = event by t (case)
3. If `y_binary` is all-0 or all-1, AUC = 0

In [ ]:
print(f"{'t':>4}  {'included':>8}  {'controls (>t)':>13}  {'cases (<=t)':>11}  {'auc_possible':>12}")
print("-" * 58)

durations = val["disc_duration"].values
events    = val["event"].values

for t in EVALUATION_TIMES:
    mask     = ~((events == 0) & (durations <= t))
    y_binary = (durations[mask] > t).astype(int)
    controls = y_binary.sum()          # survived past t
    cases    = (1 - y_binary).sum()    # had event by t
    possible = controls > 0 and cases > 0
    marker   = "  ← DEGENERATE" if not possible else ""
    print(f"{t:>4}  {mask.sum():>8}  {controls:>13}  {cases:>11}  {str(possible):>12}{marker}")

## 5. Why ci_119 is decreasing — check score variance at the last interval

CI at time `t` uses `surv[:, t]` as the predicted score. If all patients have
`disc_duration < t`, the model learns to assign near-zero survival at `t` for everyone,
collapsing score variance and degrading the concordance index.

In [ ]:
print("Proportion of validation patients whose disc_duration < t (model predicts low surv for all):")
print()
print(f"{'t':>4}  {'frac(disc_dur < t)':>20}  {'usable for CI?':>14}")
print("-" * 44)
for t in EVALUATION_TIMES:
    frac_below = (durations < t).mean()
    usable = frac_below < 1.0 and frac_below > 0.0
    marker = "  ← ALL BELOW, score collapses" if not usable else ""
    print(f"{t:>4}  {frac_below:>20.3f}  {str(usable):>14}{marker}")

## 6. Summary and suggested fix

This cell prints a recommendation based on the findings above.

In [ ]:
# Find the last evaluation time that still has controls
good_eval_times = []
for t in EVALUATION_TIMES:
    mask     = ~((events == 0) & (durations <= t))
    y_binary = (durations[mask] > t).astype(int)
    if y_binary.sum() > 0 and (1 - y_binary).sum() > 0:
        good_eval_times.append(t)

bad_eval_times = [t for t in EVALUATION_TIMES if t not in good_eval_times]
print(f"Good evaluation times (non-degenerate AUC): {good_eval_times}")
print(f"Degenerate evaluation times:                {bad_eval_times}")
print()
print("Suggested fix:")
print(f"  Remove {bad_eval_times} from evaluation_times in the config.")
print(f"  New evaluation_times: {good_eval_times}")
print()
# Also suggest the last valid interval
last_with_both = stats_df[(stats_df["n_events"] > 0) & (stats_df["n_patients"] > stats_df["n_events"])]["interval"].max()
print(f"  Last interval with both events and censored patients: {last_with_both}")
print(f"  Consider replacing 119 with {last_with_both} if you want a 'last time point' metric.")

## 7. Cross-check: raw first_event distribution from JSONL

Confirms that the issue is not in the JSONL data itself but in how discretization
interacts with the administrative censoring horizon.

In [ ]:
jsonl_durations, jsonl_events = [], []
with open(JSONL_DIR / "validation.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        fe = rec["smart"].get("first_event")
        cd = rec["smart"].get("cd_event", 0)
        if fe is not None:
            jsonl_durations.append(fe)
            jsonl_events.append(cd)

jsonl_durations = np.array(jsonl_durations)
jsonl_events    = np.array(jsonl_events)

print(f"JSONL validation: {len(jsonl_durations):,} patients")
print(f"  first_event: min={jsonl_durations.min():.0f}  max={jsonl_durations.max():.0f}")
print(f"  event rate:  {jsonl_events.mean():.3f}")
print()

# Show how many raw patients fall beyond the horizon
beyond = (jsonl_durations > HORIZON_DAYS).sum()
at     = (jsonl_durations == HORIZON_DAYS).sum()
print(f"  Patients with first_event > {HORIZON_DAYS}:  {beyond} (will be censored at horizon)")
print(f"  Patients with first_event == {HORIZON_DAYS}: {at}")

fig, ax = plt.subplots(figsize=(10, 3))
for ev, label, color in [(1, "event", "tomato"), (0, "censored", "steelblue")]:
    m = jsonl_events == ev
    ax.hist(jsonl_durations[m], bins=60, alpha=0.6, label=f"{label} (n={m.sum()})", color=color)
ax.axvline(HORIZON_DAYS, color="black", linestyle="--", label=f"horizon={HORIZON_DAYS}")
ax.set_title("Validation JSONL: raw first_event distribution")
ax.set_xlabel("days")
ax.legend()
plt.tight_layout()
plt.show()